# Step 01 — Data Ingestion

This notebook provides a visual overview of the raw macro data ingested by
`pipelines/01_ingest.py`. It loads `data/raw/macro_raw.parquet` and produces
coverage plots, sample time-series, and summary statistics.

**Run `python pipelines/01_ingest.py` before executing this notebook.**

## Setup & Imports

In [ ]:
import sys
sys.path.insert(0, "../src")

import logging

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from market_regime.config import load, setup_logging
from market_regime.runtime import RunConfig
from market_regime import DATA_DIR
from market_regime import plotting

setup_logging("INFO")
log = logging.getLogger("01_ingestion")

cfg = load()

run_cfg = RunConfig(generate_plots=True, save_plots=True, show_plots=True)

print("RunConfig:", run_cfg)
print("DATA_DIR: ", DATA_DIR)

## Load Raw Macro Data

The raw dataset is produced by `pipelines/01_ingest.py`. It contains all
multpl.com series and FRED series, resampled to quarterly frequency and
aligned to a common date index.

In [ ]:
RAW_PATH = DATA_DIR / "raw" / "macro_raw.parquet"

try:
    raw = pd.read_parquet(RAW_PATH)
    log.info("Loaded raw data: shape=%s", raw.shape)
except FileNotFoundError:
    print(f"ERROR: Raw data file not found at {RAW_PATH}")
    print("Run 'python pipelines/01_ingest.py' first.")
    raw = None
except Exception as exc:
    print(f"ERROR loading raw data: {exc}")
    raw = None

## Shape, Date Range, and Column Names

In [ ]:
if raw is not None:
    print(f"Shape:      {raw.shape[0]} quarters x {raw.shape[1]} series")
    print(f"Date range: {raw.index.min().date()} to {raw.index.max().date()}")
    print()
    print(f"Columns ({len(raw.columns)} total):")
    for col in sorted(raw.columns):
        print(f"  {col}")

## Raw Series Coverage Heatmap

Dark cells indicate that data is available for that quarter. Many multpl.com
series start in the 1950s–1960s; FRED series have varying start dates.
The gap-filling step in the feature pipeline handles interior NaNs.

In [ ]:
if raw is not None:
    plotting.plot_raw_series_coverage(raw, run_cfg)

## Sample Series Time Charts

A handful of key series — S&P 500, US inflation, 10-year Treasuries, nominal
GDP, and dividend yield — for a quick visual sanity check of the ingested data.

In [ ]:
if raw is not None:
    SAMPLE_SERIES = ["sp500", "us_infl", "10yr_ustreas", "fred_gdp", "div_yield"]
    available = [s for s in SAMPLE_SERIES if s in raw.columns]
    missing   = [s for s in SAMPLE_SERIES if s not in raw.columns]
    if missing:
        print(f"Note: series not found in raw data: {missing}")
    if available:
        plotting.plot_raw_series_sample(raw, available, run_cfg)
    else:
        print("None of the requested sample series are present.")

## Coverage Statistics

Percentage of non-NaN observations per series, sorted descending.
Series with low coverage may introduce gaps that need to be filled
in the feature engineering step.

In [ ]:
if raw is not None:
    coverage_pct = raw.notna().mean().sort_values(ascending=False) * 100
    coverage_df = coverage_pct.rename("coverage_pct").to_frame()
    coverage_df["n_valid"] = raw.notna().sum()
    coverage_df["n_total"] = len(raw)
    print(f"Total quarters: {len(raw)}")
    print()
    display(coverage_df.style.format({"coverage_pct": "{:.1f}%"}))

## First and Last 5 Rows

In [ ]:
if raw is not None:
    print("=== First 5 rows ===")
    display(raw.head())

In [ ]:
if raw is not None:
    print("=== Last 5 rows ===")
    display(raw.tail())